In [1]:
import pandas as pd
import numpy as np
import category_encoders as ce
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import joblib

# Zenginleştirilmiş MVP verisini oku
df = pd.read_parquet("../data/processed/transaction_features_mvp_sample.parquet")

# Modelleme için en güçlü kolonları seçiyoruz (Yeni hız özellikleri dahil!)
feature_cols = [
    'amount', 'tx_count_last_24h', 'sec_since_last_tx', 'speed_alert',
    'current_age', 'credit_score', 'yearly_income', 'total_debt',
    'merchant_city', 'mcc', 'use_chip', 'gender', 'is_weekend', 'hour'
]

X = df[feature_cols]
y = df['is_fraud']

# Train-Test Split (Sızıntıyı önlemek için Target Encoding öncesi bölüyoruz)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [2]:
# Yüksek kardinaliteli kolonlar için Encoder
target_enc = ce.TargetEncoder(cols=['merchant_city', 'mcc'])

# Train verisiyle öğren ve uygula
X_train = target_enc.fit_transform(X_train, y_train)
# Test verisine sadece uygula
X_test = target_enc.transform(X_test)

# Kalan az seçenekli kategorikleri (use_chip, gender) hızlıca sayıya çevir
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Sütunları eşitle (Test setinde eksik kategori kalmasın)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"✅ Preprocessing tamamlandı. Yeni sütun sayısı: {X_train.shape[1]}")

✅ Preprocessing tamamlandı. Yeni sütun sayısı: 2362


In [3]:
# Negatif / Pozitif oranı
spw = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_dream = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=spw,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

xgb_dream.fit(X_train, y_train)
print("✅ XGBoost Dream Model eğitildi.")

/Users/emre/.pyenv/versions/3.12.9/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [01:40:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✅ XGBoost Dream Model eğitildi.


In [6]:
# Olasılıkları al
y_probs = xgb_dream.predict_proba(X_test)[:, 1]

# Farklı bir threshold deneyelim (Örn: 0.4)
custom_threshold = 0.6
y_pred_custom = (y_probs >= custom_threshold).astype(int)

print(f"--- Threshold: {custom_threshold} için Sonuçlar ---")
print(classification_report(y_test, y_pred_custom))
print(f"ROC-AUC: {roc_auc_score(y_test, y_probs):.4f}")

--- Threshold: 0.6 için Sonuçlar ---
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     53329
           1       0.66      0.94      0.78      2666

    accuracy                           0.97     55995
   macro avg       0.83      0.96      0.88     55995
weighted avg       0.98      0.97      0.98     55995

ROC-AUC: 0.9885


In [7]:
# Her şeyi tek bir bundle (paket) olarak kaydet
model_bundle = {
    'model': xgb_dream,
    'encoder': target_enc,
    'features': X_train.columns.tolist()
}

joblib.dump(model_bundle, "../models/fraud_dream_model_bundle.joblib")
print("🚀 Dream Model Paketi Başarıyla Kaydedildi!")

🚀 Dream Model Paketi Başarıyla Kaydedildi!
